## Ch10. Building Neural Networks with PyTorch

#### PyTorch란?
Meta AI가 개발한 오픈소스 딥러닝 라이브러리

#### PyTorch 기초: 텐서(Tensor)
텐서란 Pytorch의 핵심 데이터 구조이다. 모양과 데이터 타입을 가진 다차원 배열로, 수치 계산에 사용한다.
NumPy 배열과 겉보기엔 동일해 보이지만, 추가로 GPU에 올려서 연산 가능하고, 자동 미분을 지원한다.
앞으로 만들 모든 신경망은 텐서를 입력받고 텐서를 출력한다.

In [35]:
# 텐서 생성
import torch

X = torch.tensor([[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]])
print(X)

tensor([[1., 4., 7.],
        [2., 3., 6.]])


In [36]:
print(X.shape)
print(X.dtype)
print(X[0, 1])
print(X[:, 1])

print(10 * (X + 1.0)) # 원소별 연산
print(X.exp()) # 지수
print(X.mean()) # 평균
print(X.max(dim=0)) # 축 별 최댓값
print(X @ X.T) # 행렬곱

torch.Size([2, 3])
torch.float32
tensor(4.)
tensor([4., 3.])
tensor([[20., 50., 80.],
        [30., 40., 70.]])
tensor([[   2.7183,   54.5981, 1096.6332],
        [   7.3891,   20.0855,  403.4288]])
tensor(3.8333)
torch.return_types.max(
values=tensor([2., 4., 7.]),
indices=tensor([1, 0, 0]))
tensor([[66., 56.],
        [56., 49.]])


In [37]:
import numpy as np

print(X.numpy()) # 텐서 -> NumPy

print(torch.tensor(np.array([[1., 4., 7.], [2., 3., 6.]]))) # NumPy -> 텐서

print(torch.FloatTensor(np.array([[1., 4., 7.], [2., 3., 6]]))) # 복사
# 제자리(in-place) 연산
X[:, 1] = -99
print(X)

X.relu_() # 음수를 0으로
print(X)

[[1. 4. 7.]
 [2. 3. 6.]]
tensor([[1., 4., 7.],
        [2., 3., 6.]], dtype=torch.float64)
tensor([[1., 4., 7.],
        [2., 3., 6.]])
tensor([[  1., -99.,   7.],
        [  2., -99.,   6.]])
tensor([[1., 0., 7.],
        [2., 0., 6.]])


#### 하드웨어 가속

In [38]:
# 디바이스 선택 코드
if torch.cuda.is_available():
	device = "cuda"
elif torch.backends.mps.is_available():
	device = "mps"
else:
	device = "cpu"

In [39]:
# 텐서를 GPU에 올리는 두 가지 방법

# 방법 1: CPU에서 만든 뒤 복사
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
M = M.to(device)
# 방법 2: 처음부터 GPU에 생성
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]], device=device)
# GPU에서 실행, 결과 R도 GPU에 존재
R = M @ M.T

print(R)

tensor([[14., 32.],
        [32., 77.]])


In [40]:
M = torch.rand((1000, 1000))
%timeit M @ M.T
M = torch.rand((1000, 1000), device=device)
%timeit M @ M.T

3.92 ms ± 41.1 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
The slowest run took 13.34 times longer than the fastest. This could mean that an intermediate result is being cached.
14.9 ms ± 15.1 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


#### Autograd(역방향 자동 미분)

In [41]:
x = torch.tensor(5.0, requires_grad=True)
f = x ** 2 # 순전파
print(f)
f.backward() # f에서 시작해 leaf node(x)까지 역전파
print(x.grad) # 역전파로 계산된 값

# 경사하강법 방법1:
learning_rate = 0.1
with torch.no_grad():
	x -= learning_rate * x.grad
# 방법2:
x_detached = x.detach()
x_detached -= learning_rate * x.grad

x.grad.zero_()
# 전체 학습 루프
x = torch.tensor(5.0, requires_grad=True)
for iteration in range(100):
	f = x ** 2
	f.backward()
	with torch.no_grad():
		x -= learning_rate * x.grad

	x.grad.zero_() # 초기화

tensor(25., grad_fn=<PowBackward0>)
tensor(10.)


In [ ]:
# in-place 연산이 autograd를 어떻게 망가뜨리나

t = torch.tensor(2.0, requires_grad=True)
z = t.exp()
z += 1 # in-place 연산
# z.backward() # RuntimeError

RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.FloatTensor []], which is output 0 of Exp, is at version 1; expected version 0 instead. Hint: enable anomaly detection to find the operation that failed to compute its gradient, with torch.autograd.set_detect_anomaly(True, check_nan=False).

#### 선형 회귀 구현

1. 데이터 준비
2. 파라미터 초기화
3. 학습 루프
4. 추론

In [44]:
# 1. 데이터 준비

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
X, y = housing.data, housing.target

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_val, y_train_val, test_size=0.2, random_state=42
)

X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)
means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
# 표준화
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds
# 타깃(y) 형태 맞추기
y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)

In [45]:
# 2. 파라미터 초기화

torch.manual_seed(42)
n_features = X_train.shape[1]
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

In [46]:
# 학습 루프(배치 경사하강법)

learning_rate = 0.4
n_epochs = 20
for epoch in range(n_epochs):
	y_pred = X_train @ w + b
	loss = ((y_pred - y_train) ** 2).mean()
	loss.backward()
	with torch.no_grad():
		b -= learning_rate * b.grad
		w -= learning_rate * w.grad
		b.grad.zero_()
		w.grad.zero_()
	print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

Epoch 1/20, Loss: 16.0495548248291
Epoch 2/20, Loss: 4.726524829864502
Epoch 3/20, Loss: 2.1483235359191895
Epoch 4/20, Loss: 1.261756420135498
Epoch 5/20, Loss: 0.9224867224693298
Epoch 6/20, Loss: 0.781599223613739
Epoch 7/20, Loss: 0.7158316969871521
Epoch 8/20, Loss: 0.6797970533370972
Epoch 9/20, Loss: 0.6563632488250732
Epoch 10/20, Loss: 0.6388936638832092
Epoch 11/20, Loss: 0.6247158646583557
Epoch 12/20, Loss: 0.6126792430877686
Epoch 13/20, Loss: 0.6022301912307739
Epoch 14/20, Loss: 0.5930591225624084
Epoch 15/20, Loss: 0.5849624276161194
Epoch 16/20, Loss: 0.5777890682220459
Epoch 17/20, Loss: 0.5714178681373596
Epoch 18/20, Loss: 0.5657476186752319
Epoch 19/20, Loss: 0.5606918931007385
Epoch 20/20, Loss: 0.5561760663986206


In [47]:
X_new = X_test[:3]

with torch.no_grad():
	y_pred = X_new @ w + b

print(y_pred)

tensor([[0.9126],
        [1.6173],
        [2.6462]])


#### PyTorch 고수준 API로 선형 회귀 구현

In [48]:
import torch.nn as nn

torch.manual_seed(42)
model = nn.Linear(in_features=n_features, out_features=1)
# 모델 파라미터
print(model.bias)
print(model.weight)

Parameter containing:
tensor([0.3117], requires_grad=True)
Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)


In [49]:
for param in model.parameters():
	[...]

In [50]:
model(X_train[:2]) # 모듈을 함수처럼 호출하면 내부적으로 forward()메서드가 실행된다.

tensor([[0.0786],
        [0.1613]], grad_fn=<AddmmBackward0>)

In [51]:
# 옵티마이저와 손실함수
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) # 옵티마이저: 파라미터 업데이트 담당
mse = nn.MSELoss()

In [52]:
# 학습함수
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
	for epoch in range(n_epochs):
		y_pred = model(X_train) # 순전파
		loss = criterion(y_pred, y_train) # 순전파
		loss.backward() # 역전파
		optimizer.step() # 파라미터 업데이트
		optimizer.zero_grad() # 초기화
		print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 4.270204544067383
Epoch 2/20, Loss: 0.7620967626571655
Epoch 3/20, Loss: 0.6113183498382568
Epoch 4/20, Loss: 0.5919578075408936
Epoch 5/20, Loss: 0.581246018409729
Epoch 6/20, Loss: 0.5727843046188354
Epoch 7/20, Loss: 0.5656700134277344
Epoch 8/20, Loss: 0.5595568418502808
Epoch 9/20, Loss: 0.5542512536048889
Epoch 10/20, Loss: 0.5496227145195007
Epoch 11/20, Loss: 0.5455722808837891
Epoch 12/20, Loss: 0.5420194864273071
Epoch 13/20, Loss: 0.5388972163200378
Epoch 14/20, Loss: 0.5361485481262207
Epoch 15/20, Loss: 0.5337244868278503
Epoch 16/20, Loss: 0.5315828919410706
Epoch 17/20, Loss: 0.5296878814697266
Epoch 18/20, Loss: 0.5280079245567322
Epoch 19/20, Loss: 0.5265161991119385
Epoch 20/20, Loss: 0.5251892805099487


In [53]:
X_new = X_test[:3]
with torch.no_grad():
	y_pred = model(X_new)

print(y_pred)

tensor([[0.8241],
        [1.6844],
        [2.6655]])


#### 회귀 MLP 구현

In [54]:
# 고수준: nn.Sequential
torch.manual_seed(42)
model = nn.Sequential(
	nn.Linear(n_features, 50),
	nn.ReLU(),
	nn.Linear(50, 40),
	nn.ReLU(),
	nn.Linear(40, 1)
)
# 학습
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs) # BGD방식으로 큰 데이터셋, 큰 모델 확장 시, 한계가 있다.

Epoch 1/20, Loss: 4.971917629241943
Epoch 2/20, Loss: 2.0490024089813232
Epoch 3/20, Loss: 1.005563735961914
Epoch 4/20, Loss: 0.8661571741104126
Epoch 5/20, Loss: 0.7838791608810425
Epoch 6/20, Loss: 0.732201874256134
Epoch 7/20, Loss: 0.6987429261207581
Epoch 8/20, Loss: 0.6764352321624756
Epoch 9/20, Loss: 0.6608064770698547
Epoch 10/20, Loss: 0.6491540670394897
Epoch 11/20, Loss: 0.6398534774780273
Epoch 12/20, Loss: 0.6319313049316406
Epoch 13/20, Loss: 0.6248723864555359
Epoch 14/20, Loss: 0.6183494925498962
Epoch 15/20, Loss: 0.6122236847877502
Epoch 16/20, Loss: 0.6063849329948425
Epoch 17/20, Loss: 0.6007852554321289
Epoch 18/20, Loss: 0.5953795909881592
Epoch 19/20, Loss: 0.5901413559913635
Epoch 20/20, Loss: 0.5850571990013123


#### DataLoaders를 사용한 미니배치 경사 하강법 구현

BGD 방식에 비해 에포크당 시간은 훨씬 느리지만, 같은 에포크에서 훨씬 낮은 loss를 기대할 수 있다.

In [56]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# 모델을 GPU로 이동
torch.manual_seed(42)
# model = nn.Sequential([...])
model = model.to(device)

In [57]:
# 미니배치 학습 함수
def train(model, optimizer, criterion, train_loader, n_epochs):
	model.train()
	for epoch in range(n_epochs):
		total_loss = 0.
		for X_batch, y_batch in train_loader:
			X_batch, y_batch = X_batch.to(device), y_batch.to(device)
			y_pred = model(X_batch)
			loss = criterion(y_pred, y_batch)
			total_loss += loss.item()
			loss.backward()
			optimizer.step()
			optimizer.zero_grad()
		mean_loss = total_loss / len(train_loader)
		print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1/20, Loss: 0.4812
Epoch 2/20, Loss: 0.4156
Epoch 3/20, Loss: 0.4331
Epoch 4/20, Loss: 0.3989
Epoch 5/20, Loss: 0.4016
Epoch 6/20, Loss: 0.3986
Epoch 7/20, Loss: 0.3735
Epoch 8/20, Loss: 0.3532
Epoch 9/20, Loss: 0.3371
Epoch 10/20, Loss: 0.3266
Epoch 11/20, Loss: 0.3197
Epoch 12/20, Loss: 0.3132
Epoch 13/20, Loss: 0.3159
Epoch 14/20, Loss: 0.3055
Epoch 15/20, Loss: 0.3015
Epoch 16/20, Loss: 0.3046
Epoch 17/20, Loss: 0.2991
Epoch 18/20, Loss: 0.2927
Epoch 19/20, Loss: 0.4031
Epoch 20/20, Loss: 0.5631


#### 모델 평가

In [58]:
def evaluate(model, data_loader, metric_fn, aggregate_fn=torch.mean):
	model.eval()
	metrics = []
	with torch.no_grad():
		for X_batch, y_batch in data_loader:
			X_batch, y_batch = X_batch.to(device), y_batch.to(device)
			y_pred = model(X_batch)
			metric = metric_fn(y_pred, y_batch)
			metrics.append(metric)
	return aggregate_fn(torch.stack(metrics))

print(type(X_valid), type(y_valid), type(y_test))
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)
print(type(X_valid), type(y_valid), type(y_test))
valid_dataset = TensorDataset(X_valid, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=32)
valid_mse = evaluate(model, valid_loader, mse)
print(valid_mse)
		

<class 'torch.Tensor'> <class 'torch.Tensor'> <class 'torch.Tensor'>
<class 'torch.Tensor'> <class 'torch.Tensor'> <class 'torch.Tensor'>
tensor(0.5381)


In [59]:
def rmse(y_pred, y_true):
	return ((y_pred - y_true) ** 2).mean().sqrt()

print(evaluate(model, valid_loader, rmse)) # 부정확한 값 - 배치마다 RMSE를 구하고, 그 값들을 평균내기 때문
print(valid_mse.sqrt()) # 정확한 값
# 올바른 방법
evaluate(model, valid_loader, mse, aggregate_fn=lambda metrics: torch.sqrt(torch.mean(metrics)))
# 배치 지표는 mse로 두고, 집계 단계에서 평균 후 제곱근 취한다.

tensor(0.7214)
tensor(0.7336)


tensor(0.7336)

In [60]:
# TorchMetrics: 배치 별로 들어오는 데이터를 누적해서, 전체 데이터 기준의 정확한 지표를 계산해주는 라이브러리

import torchmetrics

def evaluate_tm(model, data_loader, metric):
	model.eval()
	metric.reset()
	with torch.no_grad():
		for X_batch, y_batch in data_loader:
			X_batch, y_batch = X_batch.to(device), y_batch.to(device)
			y_pred = model(X_batch)
			metric.update(y_pred, y_batch)
	return metric.compute()

rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
evaluate_tm(model, valid_loader, rmse)

tensor(0.7341)

#### 커스텀 모듈을 사용하여 비순차적 모델 구축하기

**Wide&Deep**: 입력의 전부 또는 일부를 출력층에 직접 연결한다.
- deep 경로: 여러 층을 통과하며 복잡한 패턴을 학습한다.
- wide 경로: 입력이 거의 그대로 출력층에 도달해 단순한 규칙을 학습한다.

In [61]:
class WideAndDeep(nn.Module):
	def __init__(self, n_features):
		super().__init__()
		self.deep_stack = nn.Sequential(
			nn.Linear(n_features, 50), nn.ReLU(),
			nn.Linear(50, 40), nn.ReLU(),
		)
		self.output_layer = nn.Linear(40 + n_features, 1)

	def forward(self, X):
		deep_output = self.deep_stack(X)
		wide_and_deep = torch.concat([X, deep_output], dim=1)
		return self.output_layer(wide_and_deep)

torch.manual_seed(42)
model = WideAndDeep(n_features).to(device)
learning_rate = 0.002

In [62]:
class WideAndDeepV2(nn.Module):
	def __init__(self, n_features):
			super().__init__()
			self.deep_stack = nn.Sequential(
				nn.Linear(n_features - 2, 50), nn.ReLU(),
				nn.Linear(50, 40), nn.ReLU(),
			)
			self.output_layer = nn.Linear(40 + 5, 1)
	def forward(self, X):
		X_wide = X[:, :5]
		X_deep = X[:, 2:]
		deep_output = self.deep_stack(X_deep)
		wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
		return self.output_layer(wide_and_deep)

#### 다중 입력 모델

입력을 하나의 텐서로 합치기 어려울 때, 필요하다.

In [63]:
class WideAndDeepV3(nn.Module):
	def __init__(self, n_features): # V2와 동일
			super().__init__()
			self.deep_stack = nn.Sequential(
				nn.Linear(n_features - 2, 50), nn.ReLU(),
				nn.Linear(50, 40), nn.ReLU(),
			)
			self.output_layer = nn.Linear(40 + 5, 1)
	def forward(self, X_wide, X_deep):
		deep_output = self.deep_stack(X_deep)
		wide_and_deep = torch.concat([X_wide, deep_output],dim=1)
		return self.output_layer(wide_and_deep)

In [64]:
# 훈련셋
train_data_wd = TensorDataset(X_train[:, :5], X_train[:, 2:], y_train)
train_loader_wd = DataLoader(train_data_wd, batch_size=32, shuffle=True)
# 검증셋
valid_data_wd = TensorDataset(X_valid[:, :5], X_valid[:, 2:], y_valid)
valid_loader_wd = DataLoader(valid_data_wd, batch_size=32)
# 테스트셋
test_data_wd = TensorDataset(X_test[:, :5], X_test[:, 2:], y_test)
test_loader_wd = DataLoader(test_data_wd, batch_size=32)

def evaluate_wd(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    model.eval()
    metrics = []
    with torch.no_grad():
        for X_batch_wide, X_batch_deep, y_batch in data_loader:
            X_batch_wide = X_batch_wide.to(device)
            X_batch_deep = X_batch_deep.to(device)
            y_batch = y_batch.to(device)
            y_pred = model(X_batch_wide, X_batch_deep)
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)
    return aggregate_fn(torch.stack(metrics))


def train_wd(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.
        for X_batch_wide, X_batch_deep, y_batch in train_loader:
            X_batch_wide = X_batch_wide.to(device)
            X_batch_deep = X_batch_deep.to(device)
            y_batch = y_batch.to(device)
            y_pred = model(X_batch_wide, X_batch_deep)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

In [65]:
class WideAndDeepDataset(torch.utils.data.Dataset):
	def __init__(self, X_wide, X_deep, y):
		self.X_wide = X_wide
		self.X_deep = X_deep
		self.y = y
	def __len__(self):
		return len(self.y)
	def __getitem__(self, idx):
		input_dict = {"X_wide": self.X_wide[idx], "X_deep": self.X_deep[idx]}
		return input_dict, self.y[idx]

# 훈련셋
train_data_named = WideAndDeepDataset(
	X_wide=X_train[:, :5], X_deep=X_train[:, 2:], y=y_train
)
train_loader_named = DataLoader(train_data_named, batch_size=32, shuffle=True)
# 검증셋
valid_data_named = WideAndDeepDataset(
	X_wide=X_valid[:, :5], X_deep=X_valid[:, 2:], y=y_valid
)
valid_loader_named = DataLoader(valid_data_named, batch_size=32)
# 테스트셋
test_data_named = WideAndDeepDataset(
	X_wide=X_test[:, :5], X_deep=X_test[:, 2:], y=y_test
)
test_loader_named = DataLoader(test_data_named, batch_size=32)

def evaluate_named(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    model.eval()
    metrics = []
    with torch.no_grad():
        for inputs, y_batch in data_loader:
            inputs = {name: X.to(device) for name, X in inputs.items()}
            y_batch = y_batch.to(device)
            y_pred = model(**inputs)
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)
    return aggregate_fn(torch.stack(metrics))


def train_named(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.
        for inputs, y_batch in train_loader:
            inputs = {name: X.to(device) for name, X in inputs.items()}
            y_batch = y_batch.to(device)
            y_pred = model(**inputs)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

#### 다중 출력 모델

같은 데이터로 여러 작업을 할 때, 작업별로 모델을 따로 만드는 것보다, 하나의 네트워크가 작업 간에 유용한 특성을 공휴하는 것이 대체로 성능이 좋아진다.
보조 출력을 달아서, 네트워크의 하위 부분이 나머지에 의존하지 않고도 스스로 유용한 것을 학습하게 강제해서 과적합 감소 효과가 있다.

In [66]:
class WideAndDeepV4(nn.Module):
	def __init__(self, n_features):
			super().__init__()
			self.deep_stack = nn.Sequential(
				nn.Linear(n_features - 2, 50), nn.ReLU(),
				nn.Linear(50, 40), nn.ReLU(),
			)
			self.aux_output_layer = nn.Linear(40, 1)
	def forward(self, X_wide, X_deep):
		deep_output = self.deep_stack(X_deep)
		wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
		main_output = self.output_layer(wide_and_deep)
		aux_output = self.aux_output_layer(deep_output)
		return main_output, aux_output

def train_v4(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.
        for inputs, y_batch in train_loader:
            y_pred, y_pred_aux = model(**inputs)
            main_loss = criterion(y_pred_aux, y_batch)
            aux_loss = criterion(y_pred_aux, y_batch)
            loss = 0.8 * main_loss + 0.2 * aux_loss
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

def evaluate_v4(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for inputs, y_batch in data_loader:
            inputs = {name: X.to(device) for name, X  in inputs.itmes()}
            y_batch = y_batch.to(device)
            y_pred, _ = model(**inputs)
            metric.update(y_pred, y_batch)
    return metric.compute()


#### PyTorch로 이미지 분류기 만들기

TorchVison 라이브러리로 데이터셋 다운로드

In [67]:
# TorchVision으로 데이터셋 불러오기
import torch
import torchvision
import torchvision.transforms.v2 as T

toTenser = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(
	root="datasets", train=True, download=True, transform=toTenser
)
test_data = torchvision.datasets.FashionMNIST(
	root="datasets", train=False, download=True, transform=toTenser
)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
	train_and_valid_data, [55_000, 5_000]
)

In [68]:
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

X_sample, y_sample = train_data[0]
print(X_sample.shape)
print(X_sample.dtype)

print(train_and_valid_data.classes[y_sample])

torch.Size([1, 28, 28])
torch.float32
Ankle boot


In [69]:
# 분류기 만들기

class ImageClassifier(nn.Module):
	def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
		super().__init__()
		self.mlp = nn.Sequential(
			nn.Flatten(),
			nn.Linear(n_inputs, n_hidden1),
			nn.ReLU(),
			nn.Linear(n_hidden1, n_hidden2),
			nn.ReLU(),
			nn.Linear(n_hidden2, n_classes)
		)
	def forward(self, X):
		return self.mlp(X)

torch.manual_seed(42)
model = ImageClassifier(n_inputs=28 * 28, n_hidden1=300, n_hidden2=100, n_classes=10)
xentropy = nn.CrossEntropyLoss()

#### 분류 손실

1. CrossEntropyLoss
다중 클래스 용도, 출력층 활성화 없음, 모델 출력 logits, 빠르고 안정적

2. LogSoftmax + NLLLoss
다중 클래스 용도, 출력층 활성화 LogSoftmax, 모델 출력 로그 확률, 약간 느림

3. BCEWithLogitsLoss
이진, 다중 레이블 용도, 출력층 활성화 없음, 모델 출력 logits, 빠르고 안정적

4. Sigmoid + BCELoss
이진, 다중 레이블 용도, 출력층 활성화 Sigmoid, 모델 출력 확률, 약간 느리고 덜 안정적

#### 분류기 학습, 평가, 예측
학습은 앞서 만든 train() 함수와 SGD 옵티마이저 그대로 사용, 손실은 xentropy
평가 지표는 TorchMetrics의 Accuracy 스트리밍 지표


In [70]:
%pip install torchmetrics
import torchmetrics

accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [72]:
model.eval()
X_new, y_new = next(iter(valid_loader))
X_new = X_new[:3].to(device)
with torch.no_grad():
	y_pred_logits = model(X_new)

y_pred = y_pred_logits.argmax(dim=1)
print(y_pred)
print([train_and_valid_data.classes[index] for index in y_pred])

tensor([5, 5, 5])
['Sandal', 'Sandal', 'Sandal']


In [74]:
import torch.nn.functional as F

y_proba = F.softmax(y_pred_logits, dim=1)
print(y_proba.round(decimals=3))

y_top4_logits, y_top4_indices = torch.topk(y_pred_logits, k=4, dim=1)
y_top4_probas = F.softmax(y_top4_logits, dim=1)
y_top4_probas.round(decimals=3)

print(y_top4_indices)

tensor([[0.1040, 0.1010, 0.0960, 0.0950, 0.1010, 0.1060, 0.1010, 0.0930, 0.0990,
         0.1030],
        [0.1020, 0.1000, 0.0990, 0.0970, 0.0930, 0.1150, 0.1000, 0.0960, 0.1000,
         0.0970],
        [0.1010, 0.1040, 0.0990, 0.0920, 0.0940, 0.1160, 0.0970, 0.0970, 0.1010,
         0.0990]])
tensor([[5, 0, 9, 1],
        [5, 0, 1, 6],
        [5, 1, 0, 8]])


#### Optuna 하이퍼파라미터 튜닝

하히퍼파라미터를 자동으로 탐색하고 싶을 때, 전용 튜닝 라이브러리인 Optuna 사용

In [76]:
# 학습
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")
# 평가
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

In [78]:
import optuna
# trial에게 하이퍼파라미터 값을 요청하고, 그값으로 모델 생성, 학습한다. 검증셋으로 평가해 지표를 반환
def objective(trial):
	learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
	n_hidden = trial.suggest_int("n_hidden", 20, 300)
	model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=n_hidden, n_hidden2=n_hidden,
	                        n_classes=10).to(device)
	optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
	xentropy = nn.CrossEntropyLoss()

	# 학습
	train(model, optimizer, xentropy, train_loader, n_epochs)
	# 검증셋 평가
	accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
	validation_accuracy = evaluate_tm(model, valid_loader, accuracy).item()

	return validation_accuracy

In [79]:
torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=5)

print(study.best_params)
print(study.best_value)

[I 2026-10-01 12:27:02,208] A new study created in memory with name: no-name-63c529e8-333b-4295-a8c2-56b9f7a26aa1


Epoch 1/20, Loss: 2.2769
Epoch 2/20, Loss: 2.2093
Epoch 3/20, Loss: 2.1164
Epoch 4/20, Loss: 1.9776
Epoch 5/20, Loss: 1.7867
Epoch 6/20, Loss: 1.5775
Epoch 7/20, Loss: 1.3979
Epoch 8/20, Loss: 1.2605
Epoch 9/20, Loss: 1.1573
Epoch 10/20, Loss: 1.0782
Epoch 11/20, Loss: 1.0162
Epoch 12/20, Loss: 0.9665
Epoch 13/20, Loss: 0.9258
Epoch 14/20, Loss: 0.8918
Epoch 15/20, Loss: 0.8629
Epoch 16/20, Loss: 0.8382
Epoch 17/20, Loss: 0.8165
Epoch 18/20, Loss: 0.7974
Epoch 19/20, Loss: 0.7803
Epoch 20/20, Loss: 0.7649


[I 2026-10-01 12:37:51,708] Trial 0 finished with value: 0.7102000117301941 and parameters: {'learning_rate': 0.00031489116479568613, 'n_hidden': 287}. Best is trial 0 with value: 0.7102000117301941.


Epoch 1/20, Loss: 1.1392
Epoch 2/20, Loss: 0.6219
Epoch 3/20, Loss: 0.5250
Epoch 4/20, Loss: 0.4826
Epoch 5/20, Loss: 0.4571
Epoch 6/20, Loss: 0.4405
Epoch 7/20, Loss: 0.4238
Epoch 8/20, Loss: 0.4121
Epoch 9/20, Loss: 0.4020
Epoch 10/20, Loss: 0.3921
Epoch 11/20, Loss: 0.3839
Epoch 12/20, Loss: 0.3742
Epoch 13/20, Loss: 0.3664
Epoch 14/20, Loss: 0.3593
Epoch 15/20, Loss: 0.3520
Epoch 16/20, Loss: 0.3458
Epoch 17/20, Loss: 0.3399
Epoch 18/20, Loss: 0.3341
Epoch 19/20, Loss: 0.3280
Epoch 20/20, Loss: 0.3227


[I 2026-10-01 12:51:35,522] Trial 1 finished with value: 0.8708000183105469 and parameters: {'learning_rate': 0.008471801418819975, 'n_hidden': 188}. Best is trial 1 with value: 0.8708000183105469.


Epoch 1/20, Loss: 2.3091
Epoch 2/20, Loss: 2.3005
Epoch 3/20, Loss: 2.2929
Epoch 4/20, Loss: 2.2859
Epoch 5/20, Loss: 2.2792
Epoch 6/20, Loss: 2.2726
Epoch 7/20, Loss: 2.2662
Epoch 8/20, Loss: 2.2598
Epoch 9/20, Loss: 2.2534
Epoch 10/20, Loss: 2.2469
Epoch 11/20, Loss: 2.2402
Epoch 12/20, Loss: 2.2332
Epoch 13/20, Loss: 2.2256
Epoch 14/20, Loss: 2.2177
Epoch 15/20, Loss: 2.2094
Epoch 16/20, Loss: 2.2006
Epoch 17/20, Loss: 2.1913
Epoch 18/20, Loss: 2.1813
Epoch 19/20, Loss: 2.1705
Epoch 20/20, Loss: 2.1592


[I 2026-10-01 12:59:01,663] Trial 2 finished with value: 0.24279999732971191 and parameters: {'learning_rate': 4.207988669606632e-05, 'n_hidden': 63}. Best is trial 1 with value: 0.8708000183105469.


Epoch 1/20, Loss: 2.3102
Epoch 2/20, Loss: 2.3072


[W 2026-10-01 13:00:03,901] Trial 3 failed with parameters: {'learning_rate': 1.7073967431528103e-05, 'n_hidden': 263} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\lbjic\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\lbjic\AppData\Local\Temp\ipykernel_553720\4252681495.py", line 12, in objective
    train(model, optimizer, xentropy, train_loader, n_epochs)
  File "C:\Users\lbjic\AppData\Local\Temp\ipykernel_553720\2695196758.py", line 8, in train
    y_pred = model(X_batch)
             ^^^^^^^^^^^^^^
  File "c:\Users\lbjic\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\module.py", line 1783, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\lbjic\AppData\Local\Programs\Python\P

KeyboardInterrupt: 

In [ ]:
# objective()가 데이터를 전역 변수로 쓰지않고, 인자로 받도록 바꾸는 방법

from functools import partial
# 1) 로더를 인자로 받음
def objective(trial, train_loader, valid_loader, n_epochs=10):
	learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
	n_hidden = trial.suggest_int("n_hidden", 20, 300)

	model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=n_hidden,
	                        n_hidden2=n_hidden, n_classes=10).to(device)
	optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
	xentropy = nn.CrossEntropyLoss()

	train(model, optimizer, xentropy, train_loader, n_epochs)

	accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)
	validation_accuracy = evaluate_tm(model, valid_loader, accuracy).item()
	return validation_accuracy

# 2) trial 하나만 받는 함수로 변환(lambda 활용)
objective_with_data = lambda trial: objective(
	trial, train_loader=train_loader, valid_loader=valid_loader
)

torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective_with_data, n_trials=5)

print(study.best_params)
print(study.best_value)

# 2) trial 하나만 받는 함수로 변환(partial) 활용)
# objective_with_data = partial(objective, train_loader=train_loader, valid_loader=valid_loader)

In [ ]:
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0,
                                     interval_steps=1)
study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)

In [ ]:
# 가지치기(pruner: 하이퍼파라미터 탐색 과정(trial) 줄이기)넣은 objective() - 탐색 시간 및 연산 절약
def objective(trial, train_loader, valid_loader, n_epochs=10):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20, 300)

    model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=n_hidden,
                            n_hidden2=n_hidden, n_classes=10).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    xentropy = nn.CrossEntropyLoss()
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

    for epoch in range(n_epochs):
        # 1 에포크 학습
        train(model, optimizer, xentropy, train_loader, n_epochs=1)

        # 검증 정확도 평가
        validation_accuracy = evaluate_tm(model, valid_loader, accuracy).item()

        # Optuna에 현재 성능 보고, 나쁘면 중단
        trial.report(validation_accuracy, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return validation_accuracy

#### PyTorch 모델 저장과 불러오기

- 방법 1. 모델 전체 저장
가장 간단하지만, 보안 문제와 취약성의 이유로 권장하지 않는 방법이다.

- 방법 2. 가중치만 저장

- 방법 3. 하이퍼파라미터까지 함께 저장
구조 정보를 같이 저장해 두면 불러올 때 구조를 따로 기억할 필요가 없다.

In [ ]:
# 방법 1.

torch.save(model, "my_fashion_mnist.pt")
loaded_model = torch.load("my_fashion_mnist.pt", weights_only=False)
loaded_model.eval()
y_pred_logits = loaded_model(X_new)

In [ ]:
# 방법 2.
torch.save(model.state_dict(), "my_fashion_mnist_weights.pt")
new_model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=300, n_hidden2=100, n_classes=10)
loaded_weights = torch.load("my_fashion_mnist_weights.pt", weights_only=True)
new_model.load_state_dict(loaded_weights)
new_model.eval()

#### PyTorch 모델 컴파일과 최적화

**TorchScript**: 모델 코드를 정적 타입의 파이썬 부분집합으로 변환한 것.
속도 향상과 이식성의 이점이 있다.

- 변환 방법1. 트레이싱
샘플 입력으로 모델을 실제로 한 번 실행하며 일어난 연산을 기록해서 TorchScript로 변환한다. 정적 모델에 적합하다.
if/match가 있으면 실제 실행된 분기만 기록되고, 반복문이 있으면 실제 실행된 횟수만큼 연산이 복사되어 기록된다는 한계가 있다.

- 변환 방법2. 스크립팅
파이썬 코드를 직접 파싱해서 변환한다. 동적 모델에 적합하다.

PyTorch 2.0 이후 TorchScript는 더 이상 활발히 개발되지 않고, 개발 초점은 torch.compile()로 이동한다.

- torch.compile
모델을 같은 파이썬 모델 그대로 두고 내부만 최적화한다. 처음 사용할 때 컴파일하고, 학습에도 사용 가능하다.

In [ ]:
# 트레이싱
torchscript_model = torch.jit.trace(model, X_new)

# 스크립팅
torchscript_model = torch.jit.script(model)

# 추가 최적화, 저장, 로드
optimized_model = torch.jit.optimize_for_inference(torchscript_model)
torchscript_model.save('my_fashion_mnist_torchscript.pt')
loaded = torch.jit.load("my_fashion_mnist_torchscript.pt")

# torch.compile()
compiled_model = torch.compile(model)

#### Exercises

1. NumPy와 비교한 PyTorch의 핵심 추가 기능
하드웨워 가속과 자동 미분

2. exp() vs exp_(), relu() vs relu_()
_이 붙은 것은 in-place 연산이다. 이 경우, 입력 텐서를 직접 수정하고, 메모리를 아낄 수 있지만,
자동 미분과정에서 backward에 필요한 값을 덮어써서 에러를 일으킬 수 있다.

3. GPU에 새 텐서를 만드는 두 가지 방법
CPU에서 만든 뒤 복사하거나, 처음부터 GPU에 생성한다.

4. 자동미분 없이 계산하는 세 가지 방법
- with torch.no_grad(): 블록 안에서 계산한다.
- x.detach()로 그래프에서 분리한 텐서를 사용한다.
- requires_grad=False인 텐서로 계산한다.

5. 첫 번째 문제 - RuntimeError 여부
- z = t.cos().exp_(): exp()는 출력을 저장해 backward에 쓴다.
exp_()가 그 출력을 in-place로 덮어쓰므로 에러가 발생한다.
- z = t.cos_().exp(): t는 leaf텐서라서 in-place 연산 금지하므로 에러 발생한다.
- z = t.exp().cos_(): cos()는 입력을 저장하는 연산인데, 그 입력이 exp()의 출력이고,
cos_()가 이를 덮어쓴다. 따라서 입력이 사라지므로 에러가 발생한다.

두 번째 문제
- 에러가 발생한다. v.cos()는 입력 v를 저장하는데, v.sin_()이 v를 덮어써서 backward 때
원래 v를 쓸 수 없기 때문이다.
- w = v.cos_() * v.sin() 로 바꾸면, 에러 없이 동작한다.
값은 cos(3)sin(3) 에서 cos(3)sin(cos(3))으로 달라진다.

6. Linear(100, 200)
- 뉴런 수: 200
- weight shape: [200,100]
- bias shape: [200]
- 입력 shape: [배치,100]
- 출력 shape: [배치,200]

7. 학습 루프의 주요 단계
- 모델을 학습모드로 전환
- 배치를 꺼내 장치로 이동
- 순전파
- 손실 계산
- 역전파
- 파라미터 업데이트
- 그라디언트 초기화

8. 옵티마이저를 모델을 GPU로 옮긴 뒤 만드는 이유
옮기기 전 만들면 상태가 CPU에 있거나 파라미터와 장치가 어긋날 수 있어서

9. GPU 학습 속도를 위한 DataLoader 옵션
- pin_memory=True: CPU->GPU 전송 가속
- num_workers: 여러 프로세스로 다음 배치 미리 준비

10. 분류 손실 함수
- CrossEntropyLoss: 다중 클래스. 모델은 로짓 출력
- LogSoftmax + NLLLoss: 다중 클래스 대안. 로그 확률이 필요하거나 손실 전에 분포를 whwjd gkf EO
- BCEWithLogitsLoss: 이진 분류, 다중 레이블. 로짓 출력
- Sigmoid + BCELoss: 이진 분류 대안. 약간 느리고 덜 안정적

11. model.train() / model.eval()이 중요한 이유
모드를 맞추지 않으면 평가 결과가 부정확하거나 불안정해지기 때문이다.

12. trace vs script
- trace: 샘플 입력으로 실제 실행하며 연산을 기록하고, 정적 모델에 적합하다.
- script: 파이썬 코드를 직접 파싱해 변환하고, 동적 모델에 적합하다.

In [ ]:
# 13. 자동미분으로 f(x,y)=sin(x^2 y)의 그라디언트 벡터 구하기

import torch

x = torch.tensor(1.2, requires_grad=True)
y = torch.tensor(3.4, requires_grad=True)

f = torch.sin(x**2 * y)
f.backward()

print(x.grad, y.grad)

In [ ]:
# 14. Dense 모듈 구현

# 방법1: nn.Linear와 nn.ReLU 조합

import torch
import torch.nn as nn

class Dense(nn.Module):
	def __init__(self, in_features, out_features):
		super().__init__()
		self.linear = nn.Linear(in_features, out_features)
		self.relu = nn.ReLU()

	def forward(self, X):
		return self.relu(self.linear(X))

# 방법2: nn.Parameter와 relu()함수
import math

class Dense(nn.Module):
	def __init__(self, in_features, out_features):
		super().__init__()
		bound = 1 / math.sqrt(in_features)
		self.weight = nn.Parameter(
			torch.empty(out_features, in_features).uniform_(-bound, bound)
		)
		self.bias = nn.Parameter(torch.empty(out_features).uniform_(-bound, bound))

	def forward(self,X):
		return torch.relu(X @ self.weight.T + self.bias)

In [ ]:
# 15. CoverType 분류 MLP 구현

import torch, torch.nn as nn
import numpy as np
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

# 데이터 로드
data = fetch_covtype()
X, y = data.data, data.target - 1

# 분할
X_train_val, X_test, y_train_val, y_test = train_test_split(
	X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
	X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val
)

# 표준화
means = X_train.mean(axis=0)
stds = X_train.std(axis=0) + 1e-8
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

# 커스텀 데이터셋
class CoverTypeDataset(Dataset):
	def __init__(self, X, y):
		self.X = torch.tensor(X, dtype=torch.float32)
		self.y = torch.tensor(y, dtype=torch.long)

	def __len__(self):
		return len(self.y)

	def __getitem__(self, index):
		return self.X[index], self.y[index]

train_data = CoverTypeDataset(X_train, y_train)
valid_data = CoverTypeDataset(X_valid, y_valid)
test_data = CoverTypeDataset(X_test, y_test)

# 데이터로더
batch_size = 512
train_loader = DataLoader(train_data, batch_size=batch_size)
valid_loader = DataLoader(valid_data, batch_size=1024)
test_loader = DataLoader(test_data, batch_size=1024)

# 모델
class Dense(nn.Module):
	def __init__(self, in_features, out_features):
		super().__init__()
		self.linear = nn.Linear(in_features, out_features)
		self.relu = nn.ReLU()

	def forward(self, X):
		return self.relu(self.linear(X))

class CoverTypeMLP(nn.Module):
	def __init__(self, n_inputs=54, n_hidden=256, n_layers=3, n_classes=7):
		super().__init__()
		layers = [Dense(n_inputs, n_hidden)]
		for _ in range(n_layers - 1):
			layers.append(Dense(n_hidden, n_hidden))
		layers.append(nn.Linear(n_hidden, n_classes))
		self.net = nn.Sequential(*layers)
	def forward(self, X):
		return self.net(X)

# 학습, 평가
device = "cuda" if torch.cuda.is_available() else "cpu"

def train_and_eval(model, optimizer, criterion, train_loader, valid_loader, n_epochs):
	acc = torchmetrics.Accuracy(task="multiclass", num_classes=7).to(device)
	for epoch in range(n_epochs):
		model.train()
		for Xb, yb in train_loader:
			Xb, yb = Xb.to(device), yb.to(device)
			loss = criterion(model(Xb), yb)
			loss.backward()
			optimizer.step()
			optimizer.zero_grad()

		valid_acc = evaluate_tm(model, valid_loader, acc).item()
		print(f"Epoch {epoch+1}/{n_epochs}, valid acc: {valid_acc:.4f}")
	return valid_acc

torch.manual_seed(42)
model = CoverTypeMLP(n_hidden=256, n_layers=3).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
criterion = nn.CrossEntropyLoss()
train_and_eval(model, optimizer, criterion, train_loader, valid_loader, n_epochs=20)

# optuna로 하이퍼파라미터 자동 탐색
def objective(trial):
	n_hidden = trial.suggest_int("n_hidden", 64, 512)
	n_layers = trial.suggest_int("n_layers", 2, 5)
	lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
	bs = trial.suggest_categorical("batch_size", [256, 512, 1024])

	loader = DataLoader(train_data, batch_size=bs, shuffle=True)
	model = CoverTypeMLP(n_hidden=n_hidden, n_layers=n_layers).to(device)
	optimizer = torch.optim.Adam(model.parameters(), lr=lr)
	return train_and_eval(model, optimizer, nn.CrossEntropyLoss(),
	                      loader, valid_loader, n_epochs=10)
study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=20)
print(study.best_params, study.best_value)

# 최종 테스트 평가
acc = torchmetrics.Accuracy(task="multiclass", num_classes=7).to(device)
print(f"test acc: {evaluate_tm(model, test_loader, acc).item()}")